# Homework 8 — Wine Quality Models

Name: Alex Devoid  
Course: ST 554

## Introduction

In this notebook I continue the wine quality analysis from homework 7. I keep the earlier linear and logistic models, then add tree-based models so I can compare everything on the same train/CV/test split.


In [1]:

import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV
from sklearn.linear_model import (
    LinearRegression,
    LassoCV,
    RidgeCV,
    ElasticNetCV,
    LogisticRegression,
    LogisticRegressionCV,
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, log_loss, accuracy_score

# clean up the notebook output
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

# fix the random seed so I can reproduce the same split and tuned models.
RANDOM_STATE = 1234


## 1. Read in and Combine Data

I read the red and white wine files separately, then stack them into one data frame and add a wine-type variable so the source of each observation stays clear after the combine step.

I also create `type_binary` as a 0/1 helper for the modeling sections, since scikit-learn needs numeric inputs once wine type enters a model.


In [2]:
# read the two source files directly from the UCI repository.
red_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
white_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

# add wine type before combining so I can keep both a readable label and a numeric version for sklearn.
red_wine = pd.read_csv(red_url, sep=';').assign(type='red', type_binary=0)
white_wine = pd.read_csv(white_url, sep=';').assign(type='white', type_binary=1)
wine = pd.concat([red_wine, white_wine], ignore_index=True)

# build a few checks before moving on to the split and modeling sections.
row_summary = pd.DataFrame(
    {
        'rows': [len(red_wine), len(white_wine), len(wine)],
        'columns': [red_wine.shape[1], white_wine.shape[1], wine.shape[1]],
    },
    index=['red_wine', 'white_wine', 'combined'],
)
row_summary.index.name = 'dataset'

type_summary = wine['type'].value_counts().rename_axis('type').to_frame('count')
type_summary['proportion'] = type_summary['count'] / len(wine)
missing_cells = int(wine.isna().sum().sum())

# show the row counts, a look at the data, and the type balance.
display(row_summary)
display(wine.head())
display(type_summary)


,rows,columns
dataset,,
red_wine,1599,14
white_wine,4898,14
combined,6497,14


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type_binary
0,7.4000,0.7000,0.0000,1.9000,0.0760,11.0000,34.0000,0.9978,3.5100,0.5600,9.4000,5,red,0
1,7.8000,0.8800,0.0000,2.6000,0.0980,25.0000,67.0000,0.9968,3.2000,0.6800,9.8000,5,red,0
2,7.8000,0.7600,0.0400,2.3000,0.0920,15.0000,54.0000,0.9970,3.2600,0.6500,9.8000,5,red,0
3,11.2000,0.2800,0.5600,1.9000,0.0750,17.0000,60.0000,0.9980,3.1600,0.5800,9.8000,6,red,0
4,7.4000,0.7000,0.0000,1.9000,0.0760,11.0000,34.0000,0.9978,3.5100,0.5600,9.4000,5,red,0


,count,proportion
type,,
white,4898,0.7539
red,1599,0.2461


The combined data set has 6,497 rows and 14 columns after adding the wine-type label and a 0/1 encoding. There are 4,898 white wines and 1,599 red wines, so the classes are not perfectly balanced. The data have 0 missing cells, so I do not need a missing-value imputation step before modeling.


## 2. Split the Data

I use the same combined wine table for both modeling sections, but the response changes by task. For the regression models, `alcohol` is the response. For the classification models, wine type is the response.

I keep the string `type` column for summaries and for the stratified split. I use `type_binary` whenever I need a numeric version of wine type inside a model.


In [3]:
# variables that show up in both modeling tasks.
base_physicochemical_features = [
    'fixed acidity',
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'total sulfur dioxide',
    'density',
    'pH',
    'sulphates',
]

# start with the shared chemistry variables, then add wine type for regression and alcohol for classification.
regression_features = base_physicochemical_features + ['type_binary']
classification_features = base_physicochemical_features + ['alcohol']

# stratify on the readable wine-type label so the train/test split preserves class balance.
train_df, test_df = train_test_split(
    wine,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=wine['type'],
)

# using a helper function here to compare type counts across the full data, training set, and test set.
def type_share(frame, label):
    # keep the red and white counts in a fixed order across all three summaries.
    counts = frame['type'].value_counts().reindex(['red', 'white'])
    # return one small table so the three summaries stack cleanly.
    return pd.DataFrame(
        {
            'split': label,
            'type': counts.index,
            'count': counts.values,
            'proportion': counts.values / len(frame),
        }
    )

# showing the three summaries stacked to check the proportions side by side.
split_summary = pd.concat(
    [
        type_share(wine, 'full_data'),
        type_share(train_df, 'training'),
        type_share(test_df, 'test'),
    ],
    ignore_index=True,
)

display(split_summary)


,split,type,count,proportion
0,full_data,red,1599,0.2461
1,full_data,white,4898,0.7539
2,training,red,1199,0.2461
3,training,white,3673,0.7539
4,test,red,400,0.2462
5,test,white,1225,0.7538


The training set has 4,872 observations and the test set has 1,625 observations. The red/white proportions stay very close across the full data, training set, and test set.


## 3. Regression Models

I fit four multiple linear regression models for `alcohol`: one full additive model, one smaller model, one model with a couple of interactions, and one model with a few polynomial terms. Then I use cross-validation on the training data to pick the plain MLR I carry forward.

After that I fit LASSO, Ridge, and Elastic Net on one common predictor set. I standardize the predictors inside each pipeline before fitting the regularized models.


In [4]:
# define four MLR candidate models to compare on the training data.
def build_regression_frames(frame):
    # use all of the shared regression predictors in the full additive model.
    additive = frame[regression_features].copy()

    # keep a smaller core model so I can compare against a reduced specification.
    reduced = frame[
        ['fixed acidity', 'volatile acidity', 'residual sugar', 'density', 'sulphates', 'type_binary']
    ].copy()

    # start from the additive model, then add two interaction terms.
    interaction = additive.copy()
    interaction['volatile acidity x sulphates'] = (
        frame['volatile acidity'] * frame['sulphates']
    )
    interaction['density x residual sugar'] = (
        frame['density'] * frame['residual sugar']
    )

    # start from the additive model again, then add a few squared terms.
    polynomial = additive.copy()
    # square the selected columns one at a time so the names stay readable.
    for col in ['fixed acidity', 'residual sugar', 'density']:
        polynomial[f'{col}^2'] = frame[col] ** 2

    # return all four model frames together so I can loop over them below.
    return {
        'additive_all': additive,
        'reduced_core': reduced,
        'interaction_core': interaction,
        'polynomial_core': polynomial,
    }

# build the training and test design matrices for the regression section.
reg_train_frames = build_regression_frames(train_df)
reg_test_frames = build_regression_frames(test_df)
y_train_reg = train_df['alcohol']
y_test_reg = test_df['alcohol']

# compare the four plain MLR candidates with 5-fold CV and summarize the RMSE values.
regression_cv_rows = []
for model_name, X_train_model in reg_train_frames.items():
    cv = cross_validate(
        LinearRegression(),
        X_train_model,
        y_train_reg,
        cv=5,
        scoring='neg_mean_squared_error',
    )
    regression_cv_rows.append(
        {
            'model': model_name,
            'cv_rmse': np.sqrt(-cv['test_score'].mean()),
        }
    )

regression_cv_summary = (
    pd.DataFrame(regression_cv_rows)
    .sort_values('cv_rmse')
    .reset_index(drop=True)
)

# refiting the plain MLR with the best CV RMSE on the full training data.
best_mlr_name = regression_cv_summary.loc[0, 'model']
best_mlr = LinearRegression().fit(reg_train_frames[best_mlr_name], y_train_reg)

# useing one common predictor set for the regularized models and let the penalty do the shrinking.
X_train_pen = train_df[regression_features]
X_test_pen = test_df[regression_features]

lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=10000)),
])
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RidgeCV(alphas=np.logspace(-3, 3, 60), cv=5, scoring='neg_mean_squared_error')),
])
elastic_net_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNetCV(
        cv=5,
        random_state=RANDOM_STATE,
        l1_ratio=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0],
        max_iter=10000,
    )),
])

# fiting the three regularized models on the training data.
for pipe in [lasso_pipe, ridge_pipe, elastic_net_pipe]:
    pipe.fit(X_train_pen, y_train_reg)

# summarize the selected tuning values so they are easy to compare.
regression_tuning_summary = pd.DataFrame(
    [
        {
            'model': 'LASSO',
            'selected_value': lasso_pipe.named_steps['model'].alpha_,
            'extra_setting': 'alpha selected by 5-fold CV',
        },
        {
            'model': 'Ridge',
            'selected_value': ridge_pipe.named_steps['model'].alpha_,
            'extra_setting': 'alpha selected by 5-fold CV',
        },
        {
            'model': 'Elastic Net',
            'selected_value': elastic_net_pipe.named_steps['model'].alpha_,
            'extra_setting': f"l1_ratio = {elastic_net_pipe.named_steps['model'].l1_ratio_}",
        },
    ]
)

# comparing the four final regression models on the held-out test set.
regression_predictions = {
    f'Best MLR ({best_mlr_name})': best_mlr.predict(reg_test_frames[best_mlr_name]),
    'LASSO': lasso_pipe.predict(X_test_pen),
    'Ridge': ridge_pipe.predict(X_test_pen),
    'Elastic Net': elastic_net_pipe.predict(X_test_pen),
}

regression_test_rows = []
for model_name, preds in regression_predictions.items():
    regression_test_rows.append(
        {
            'model': model_name,
            'rmse': np.sqrt(mean_squared_error(y_test_reg, preds)),
            'mae': mean_absolute_error(y_test_reg, preds),
        }
    )

regression_test_summary = (
    pd.DataFrame(regression_test_rows)
    .sort_values(['rmse', 'mae'])
    .reset_index(drop=True)
)

best_regression_model = regression_test_summary.loc[0, 'model']

print('Plain MLR candidates chosen with 5-fold CV on the training set')
display(regression_cv_summary)
print('Regularized regression tuning summary')
display(regression_tuning_summary)
print('Final regression comparison on the test set')
display(regression_test_summary)


Plain MLR candidates chosen with 5-fold CV on the training set


,model,cv_rmse
0,polynomial_core,0.4405
1,additive_all,0.4555
2,interaction_core,0.4557
3,reduced_core,0.5733


Regularized regression tuning summary


,model,selected_value,extra_setting
0,LASSO,0.0008,alpha selected by 5-fold CV
1,Ridge,0.0010,alpha selected by 5-fold CV
2,Elastic Net,0.0008,l1_ratio = 1.0


Final regression comparison on the test set


,model,rmse,mae
0,Best MLR (polynomial_core),0.4581,0.3317
1,LASSO,0.6377,0.3564
2,Elastic Net,0.6377,0.3564
3,Ridge,0.6387,0.3560


Among the four plain MLR candidates, `polynomial_core` had the smallest CV RMSE, so I use that model as the plain MLR benchmark. On the test set, `Best MLR (polynomial_core)` finished with the best RMSE, and the MAE column gives a second way to compare the final four models.


## 4. Classification Models

I use the same train/CV/test structure for the logistic models. I compare four plain logistic specifications built from the same feature ideas as above, choose the strongest plain logistic model with cross-validated log-loss, and then fit L1, L2, and Elastic Net logistic models with CV-based tuning.

The response is `type_binary`, where `1` means white wine and `0` means red wine.


In [5]:
# define four logistic model frames that mirror the linear-model section.
def build_classification_frames(frame):
    # use all of the shared classification predictors in the full additive model.
    additive = frame[classification_features].copy()

    # keep a smaller core model so I can compare against a reduced specification.
    reduced = frame[
        ['volatile acidity', 'residual sugar', 'chlorides', 'density', 'alcohol']
    ].copy()

    # start from the additive model, then add two interaction terms.
    interaction = additive.copy()
    interaction['volatile acidity x sulphates'] = (
        frame['volatile acidity'] * frame['sulphates']
    )
    interaction['density x alcohol'] = frame['density'] * frame['alcohol']

    # start from the additive model again, then add a few squared terms.
    polynomial = additive.copy()
    # square the selected columns one at a time so the names stay readable.
    for col in ['volatile acidity', 'density', 'alcohol']:
        polynomial[f'{col}^2'] = frame[col] ** 2

    # return all four model frames together so I can loop over them below.
    return {
        'additive_all': additive,
        'reduced_core': reduced,
        'interaction_core': interaction,
        'polynomial_core': polynomial,
    }

# build the training and test design matrices for the classification section.
class_train_frames = build_classification_frames(train_df)
class_test_frames = build_classification_frames(test_df)
y_train_cls = train_df['type_binary']
y_test_cls = test_df['type_binary']

# compare the four plain logistic candidates with 5-fold CV using log-loss.
plain_logistic_rows = []
for model_name, X_train_model in class_train_frames.items():
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000)),
    ])
    cv = cross_validate(
        pipeline,
        X_train_model,
        y_train_cls,
        cv=5,
        scoring='neg_log_loss',
    )
    plain_logistic_rows.append(
        {
            'model': model_name,
            'cv_log_loss': -cv['test_score'].mean(),
        }
    )

plain_logistic_summary = (
    pd.DataFrame(plain_logistic_rows)
    .sort_values('cv_log_loss')
    .reset_index(drop=True)
)

# refit the plain logistic model with the best CV log-loss on the full training data.
best_plain_logistic_name = plain_logistic_summary.loc[0, 'model']
best_plain_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000)),
])
best_plain_logistic.fit(class_train_frames[best_plain_logistic_name], y_train_cls)

# use one common predictor set for the regularized logistic models.
X_train_cls = train_df[classification_features]
X_test_cls = test_df[classification_features]

l1_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegressionCV(
        cv=5,
        penalty='l1',
        solver='saga',
        scoring='neg_log_loss',
        max_iter=5000,
        random_state=RANDOM_STATE,
        Cs=np.logspace(-3, 3, 10),
    )),
])

l2_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegressionCV(
        cv=5,
        penalty='l2',
        solver='lbfgs',
        scoring='neg_log_loss',
        max_iter=5000,
        random_state=RANDOM_STATE,
        Cs=np.logspace(-3, 3, 10),
    )),
])

elastic_logistic = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegressionCV(
        cv=5,
        penalty='elasticnet',
        solver='saga',
        scoring='neg_log_loss',
        max_iter=5000,
        random_state=RANDOM_STATE,
        Cs=np.logspace(-3, 3, 10),
        l1_ratios=[0.1, 0.5, 0.9],
    )),
])

# fit the three regularized logistic models on the training data.
for pipe in [l1_logistic, l2_logistic, elastic_logistic]:
    pipe.fit(X_train_cls, y_train_cls)

# summarizing the selected tuning values so the three fits are easy to compare.
classification_tuning_summary = pd.DataFrame(
    [
        {
            'model': 'L1 logistic',
            'selected_C': l1_logistic.named_steps['model'].C_[0],
            'extra_setting': 'penalty = l1',
        },
        {
            'model': 'L2 logistic',
            'selected_C': l2_logistic.named_steps['model'].C_[0],
            'extra_setting': 'penalty = l2',
        },
        {
            'model': 'Elastic Net logistic',
            'selected_C': elastic_logistic.named_steps['model'].C_[0],
            'extra_setting': f"l1_ratio = {elastic_logistic.named_steps['model'].l1_ratio_[0]}",
        },
    ]
)

# comparing the four final classification models on the held-out test set.
classification_probabilities = {
    f'Best plain logistic ({best_plain_logistic_name})': best_plain_logistic.predict_proba(class_test_frames[best_plain_logistic_name])[:, 1],
    'L1 logistic': l1_logistic.predict_proba(X_test_cls)[:, 1],
    'L2 logistic': l2_logistic.predict_proba(X_test_cls)[:, 1],
    'Elastic Net logistic': elastic_logistic.predict_proba(X_test_cls)[:, 1],
}

classification_test_rows = []
for model_name, probs in classification_probabilities.items():
    preds = (probs >= 0.5).astype(int)
    classification_test_rows.append(
        {
            'model': model_name,
            'log_loss': log_loss(y_test_cls, probs),
            'accuracy': accuracy_score(y_test_cls, preds),
        }
    )

classification_test_summary = (
    pd.DataFrame(classification_test_rows)
    .sort_values(['log_loss', 'accuracy'], ascending=[True, False])
    .reset_index(drop=True)
)

best_classification_model = classification_test_summary.loc[0, 'model']

print('Plain logistic candidates chosen with 5-fold CV on the training set')
display(plain_logistic_summary)
print('Regularized logistic tuning summary')
display(classification_tuning_summary)
print('Final classification comparison on the test set')
display(classification_test_summary)


Plain logistic candidates chosen with 5-fold CV on the training set


,model,cv_log_loss
0,polynomial_core,0.0276
1,additive_all,0.0289
2,interaction_core,0.0304
3,reduced_core,0.0514


Regularized logistic tuning summary


,model,selected_C,extra_setting
0,L1 logistic,2.1544,penalty = l1
1,L2 logistic,10.0000,penalty = l2
2,Elastic Net logistic,10.0000,l1_ratio = 0.5


Final classification comparison on the test set


,model,log_loss,accuracy
0,L2 logistic,0.0570,0.9920
1,L1 logistic,0.0570,0.9920
2,Elastic Net logistic,0.0575,0.9920
3,Best plain logistic (polynomial_core),0.0610,0.9908


The plain logistic candidate with the best training-stage CV log-loss was `polynomial_core`. On the test set, `L2 logistic` had the best log-loss, while the highest test accuracy was shared by `L2 logistic`, `L1 logistic`, and `Elastic Net logistic`.


## Homework 7 Takeaways

For the regression task, the best test-set model was `Best MLR (polynomial_core)` with an RMSE of `0.4581` and an MAE of `0.3317`. Among the regularized fits, `LASSO` and `Elastic Net` were tied to the displayed precision on RMSE at `0.6377`, while `Ridge` had a very similar RMSE and the smallest MAE of the three regularized models at `0.3560`.

For the classification task, the best model by test log-loss was `L2 logistic` with a log-loss of `0.0570`, while the highest test accuracy, `0.9920`, was shared by `L2 logistic`, `L1 logistic`, and `Elastic Net logistic`. The best log-loss model was also among the models with the best accuracy.




## 5. Tree-Based Regression Models

I keep `alcohol` as the regression response and add a regression tree and a random forest. 
I use the same train/test split from the earlier sections so the tree-based models can be compared to the regression models.


In [6]:
# reuse the same predictor regression feature set 
# so the tree models are compared on the same inputs as before.
X_train_tree_reg = train_df[regression_features]
X_test_tree_reg = test_df[regression_features]
y_train_tree_reg = train_df['alcohol']
y_test_tree_reg = test_df['alcohol']

# search over max_depth and min_samples_leaf with 5-fold CV 
# then keep the tree with the best CV squared-error score.
reg_tree_grid = {
    'max_depth': range(2, 15),
    'min_samples_leaf': [3, 5, 10, 50, 100],
}
reg_tree_tune = GridSearchCV(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    reg_tree_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)
reg_tree_tune.fit(X_train_tree_reg, y_train_tree_reg)
best_reg_tree = reg_tree_tune.best_estimator_

# search over max_features for the random forest 
# while holding the number of trees fixed at 500.
reg_forest_grid = {
    'max_features': [2, 4, 6, 8, len(regression_features)],
}
reg_forest_tune = GridSearchCV(
    RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
    reg_forest_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
)
reg_forest_tune.fit(X_train_tree_reg, y_train_tree_reg)
best_reg_forest = reg_forest_tune.best_estimator_

# collect the best CV settings 
# and the corresponding CV RMSE in one summary table.
regression_tree_tuning_summary = pd.DataFrame(
    [
        {
            'model': 'Regression tree',
            'best_cv_rmse': np.sqrt(-reg_tree_tune.best_score_),
            'max_depth': reg_tree_tune.best_params_['max_depth'],
            'min_samples_leaf': reg_tree_tune.best_params_['min_samples_leaf'],
            'max_features': np.nan,
        },
        {
            'model': 'Random forest',
            'best_cv_rmse': np.sqrt(-reg_forest_tune.best_score_),
            'max_depth': np.nan,
            'min_samples_leaf': np.nan,
            'max_features': reg_forest_tune.best_params_['max_features'],
        },
    ]
)

# generate test-set predictions from the tuned tree models
# then compare them to the regression models.
regression_tree_predictions = {
    'Regression tree': best_reg_tree.predict(X_test_tree_reg),
    'Random forest': best_reg_forest.predict(X_test_tree_reg),
}

# store the test-set metrics for the two new tree-based regression models.
regression_tree_rows = []
for model_name, preds in regression_tree_predictions.items():
    regression_tree_rows.append(
        {
            'model': model_name,
            'rmse': np.sqrt(mean_squared_error(y_test_tree_reg, preds)),
            'mae': mean_absolute_error(y_test_tree_reg, preds),
        }
    )

# turn the tree-model metrics into a results table.
regression_tree_test_summary = pd.DataFrame(regression_tree_rows)
# stack the regression results 
regression_all_test_summary = (
    pd.concat([regression_test_summary.copy(), regression_tree_test_summary], ignore_index=True)
    # sort from best to worst.
    .sort_values(['rmse', 'mae'])
    .reset_index(drop=True)
)

print('Tree-based regression tuning summary')
display(regression_tree_tuning_summary)
print('Full regression comparison including homework 7 models')
display(regression_all_test_summary)


Tree-based regression tuning summary


,model,best_cv_rmse,max_depth,min_samples_leaf,max_features
0,Regression tree,0.5493,14.0000,10.0000,NaN
1,Random forest,0.4065,NaN,NaN,8.0000


Full regression comparison including homework 7 models


,model,rmse,mae
0,Random forest,0.3905,0.2671
1,Best MLR (polynomial_core),0.4581,0.3317
2,Regression tree,0.5280,0.3812
3,LASSO,0.6377,0.3564
4,Elastic Net,0.6377,0.3564
5,Ridge,0.6387,0.3560


The regression tree chose `max_depth = 14` and `min_samples_leaf = 10`, while the random forest chose `max_features = 8`. On the full test-set comparison, the random forest finished first with an RMSE of `0.3905` and an MAE of `0.2671`, so it improved on the homework 7 regression models. The single regression tree beat the penalized linear models on RMSE, but not on MAE, and it did not beat the best plain MLR from homework 7.

## 6. Tree-Based Classification Models

I keep wine type as the response and add a classification tree and a random forest classifier. I use log-loss during training and then compare the final models on both log-loss and accuracy.

I use the same train/test split as before so the tree-based classifiers can be compared directly to the homework 7 classification models.


In [7]:
# reuse the same classification predictor set so the test comparison stays direct.
X_train_tree_cls = train_df[classification_features]
X_test_tree_cls = test_df[classification_features]
y_train_tree_cls = train_df['type_binary']
y_test_tree_cls = test_df['type_binary']

# tune the classification tree
cls_tree_grid = {
    'max_depth': range(2, 15),
    'min_samples_leaf': [3, 5, 10, 50, 100],
}
cls_tree_tune = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    cls_tree_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1,
)
cls_tree_tune.fit(X_train_tree_cls, y_train_tree_cls)
best_cls_tree = cls_tree_tune.best_estimator_

# tune max_features for the random forest classifier.
cls_forest_grid = {
    'max_features': [2, 4, 6, 8, len(classification_features)],
}
cls_forest_tune = GridSearchCV(
    RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1),
    cls_forest_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1,
)
cls_forest_tune.fit(X_train_tree_cls, y_train_tree_cls)
best_cls_forest = cls_forest_tune.best_estimator_

# collect the best tuning values in one table.
classification_tree_tuning_summary = pd.DataFrame(
    [
        {
            'model': 'Classification tree',
            'best_cv_log_loss': -cls_tree_tune.best_score_,
            'max_depth': cls_tree_tune.best_params_['max_depth'],
            'min_samples_leaf': cls_tree_tune.best_params_['min_samples_leaf'],
            'max_features': np.nan,
        },
        {
            'model': 'Random forest classifier',
            'best_cv_log_loss': -cls_forest_tune.best_score_,
            'max_depth': np.nan,
            'min_samples_leaf': np.nan,
            'max_features': cls_forest_tune.best_params_['max_features'],
        },
    ]
)

# compare the two new classifiers to the homework 7 classification models on the test set.
classification_tree_probabilities = {
    'Classification tree': best_cls_tree.predict_proba(X_test_tree_cls)[:, 1],
    'Random forest classifier': best_cls_forest.predict_proba(X_test_tree_cls)[:, 1],
}

# store the test-set metrics for the two HW8 tree-based classification models.
classification_tree_rows = []
for model_name, probs in classification_tree_probabilities.items():
    preds = (probs >= 0.5).astype(int)
    classification_tree_rows.append(
        {
            'model': model_name,
            'log_loss': log_loss(y_test_tree_cls, probs),
            'accuracy': accuracy_score(y_test_tree_cls, preds),
        }
    )

# turn the tree-model metrics into a results table.
classification_tree_test_summary = pd.DataFrame(classification_tree_rows)
# stack the HW7 and HW8 classification results.
classification_all_test_summary = (
    pd.concat([classification_test_summary.copy(), classification_tree_test_summary], ignore_index=True)
    # sort by log-loss and accuracy
    .sort_values(['log_loss', 'accuracy'], ascending=[True, False])
    .reset_index(drop=True)
)

print('Tree-based classification tuning summary')
display(classification_tree_tuning_summary)
print('Full classification comparison including homework 7 models')
display(classification_all_test_summary)


Tree-based classification tuning summary


,model,best_cv_log_loss,max_depth,min_samples_leaf,max_features
0,Classification tree,0.0906,4.0000,100.0000,NaN
1,Random forest classifier,0.0400,NaN,NaN,2.0000


Full classification comparison including homework 7 models


,model,log_loss,accuracy
0,Random forest classifier,0.0362,0.9932
1,L2 logistic,0.0570,0.9920
2,L1 logistic,0.0570,0.9920
3,Elastic Net logistic,0.0575,0.9920
4,Best plain logistic (polynomial_core),0.0610,0.9908
5,Classification tree,0.1038,0.9563


The classification tree chose `max_depth = 4` and `min_samples_leaf = 100`, while the random forest classifier chose `max_features = 2`. On the full test-set comparison, the random forest classifier finished first with a log-loss of `0.0362` and an accuracy of `0.9932`, so it improved on the homework 7 classifiers. The single classification tree had the largest log-loss and the lowest accuracy.

## Final Takeaways

For the regression task, the random forest gave the best overall test-set performance with an RMSE of `0.3905` and an MAE of `0.2671`. That was better than the homework 7 models, like the `Best MLR (polynomial_core)`, which finished at `0.4581` RMSE and `0.3317` MAE. The single regression tree was between the best plain MLR and the penalized linear models: it beat the penalized fits on RMSE, but not on MAE.

For the classification task, the random forest classifier also gave the best overall results with a log-loss of `0.0362` and an accuracy of `0.9932`. In the comparison table, `L2 logistic` is first among the homework 7 models at `0.0570` log-loss and `0.9920` accuracy. The forest improved both metrics. The single classification tree was last in the full comparison table.

Overall, the random forests gave the strongest predictive performance.